# ROS Model Training Pipeline

**Purpose:** Train Rest of Season (ROS) prediction models from historical data

**Last Updated:** 2025-10-08

---

## Overview
This notebook trains the complete ROS ensemble system:
- **DirectROSForecaster** (50% weight): Multi-quantile model with time series lags + all features
- **DartsTemporalEnsemble** (40% weight): TCN + TSMixer + AutoARIMA on pure WAR trajectories
- **Baseline MultiQuantileHistGB** (10% weight): Pure feature-based quantile regression

## Training Data Structure (Multipoint Splits)
Historical full seasons (2016-2024) are split into **multiple** training samples per season:
- **25% split**: Stats through ~40 games → predict remaining 122 games WAR
- **50% split**: Stats through ~81 games → predict remaining 81 games WAR
- **75% split**: Stats through ~121 games → predict remaining 41 games WAR

This creates 3x more training data and teaches the model about **season timing** (early-season SSS vs late-season).

## Key Innovation: Flexible Timing
The `season_completion_pct` feature allows projections at **any point** in the season:
- Not limited to fixed firsthalf/secondhalf splits
- Works for 3 weeks into season (9%), All-Star break (50%), or any custom point
- Model learns timing effects from multiple split points and interpolates

In [1]:
# Cell 1: Imports and Setup

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
from datetime import datetime

# Add project root to path
project_root = Path('.').absolute().parent.parent
sys.path.insert(0, str(project_root))

from new_pipeline.models.ros import (
    HitterROSEnsemble,
    PitcherROSEnsemble,
    ROS_HITTER_FEATURES,
    ROS_PITCHER_FEATURES,
    prepare_ros_training_data,
    temporal_cv_split,
    calculate_ros_metrics
)
from new_pipeline.common.features import ROSFeatureBuilder
from new_pipeline.common.data_preparation import create_multipoint_splits
from new_pipeline.notebooks.shared.pipeline_runner import load_historical_data

print("Imports successful!")
print(f"ROS Hitter Features: {len(ROS_HITTER_FEATURES)}")
print(f"ROS Pitcher Features: {len(ROS_PITCHER_FEATURES)}")

17:25:03 - new_pipeline.common.logging_config - INFO - Logging module initialized for new_pipeline
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\fs\__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports successful!
ROS Hitter Features: 66
ROS Pitcher Features: 52


In [2]:
# Cell 2: Load and Process Historical Data (2016-2024)

print("="*70)
print("LOADING HISTORICAL FULL-SEASON DATA (2016-2024)")
print("="*70)

from new_pipeline.notebooks.shared.pipeline_runner import run_data_pipeline

# Load raw historical data
print("\nLoading raw hitter data...")
hitter_raw = load_historical_data(
    player_type='hitter',
    years=range(2016, 2025)  # 2016-2024
)

print("\nLoading raw pitcher data...")
pitcher_raw = load_historical_data(
    player_type='pitcher',
    years=range(2016, 2025)  # 2016-2024
)

print(f"\nLoaded {len(hitter_raw)} hitter seasons (raw)")
print(f"Loaded {len(pitcher_raw)} pitcher seasons (raw)")

# Process through pipeline (adds features + Age + WAR_per_600/WAR_per_162)
print("\nProcessing hitter data through pipeline...")
print("  Pipeline adds: Age (from BP_Data), features, WAR_per_600")
hitter_processed = run_data_pipeline(hitter_raw, 'hitter')

print("\nProcessing pitcher data through pipeline...")
print("  Pipeline adds: Age (from BP_Data), features, WAR_per_162")
pitcher_processed = run_data_pipeline(pitcher_raw, 'pitcher')

print(f"\nProcessed {len(hitter_processed)} qualified hitters")
print(f"Processed {len(pitcher_processed)} qualified pitchers")
print(f"Years: {sorted(hitter_processed['Year'].unique())}")

# Verify Age and WAR rates are present
print("\nData quality checks:")
print(f"  Hitters with Age: {hitter_processed['Age'].notna().sum()}")
print(f"  Hitters with WAR_per_600: {hitter_processed['WAR_per_600'].notna().sum()}")
print(f"  Pitchers with Age: {pitcher_processed['Age'].notna().sum()}")
print(f"  Pitchers with WAR_per_162: {pitcher_processed['WAR_per_162'].notna().sum()}")

if 'Age' in hitter_processed.columns:
    print(f"\n  Hitter Age range: [{hitter_processed['Age'].min():.0f}, {hitter_processed['Age'].max():.0f}]")
    print(f"  Pitcher Age range: [{pitcher_processed['Age'].min():.0f}, {pitcher_processed['Age'].max():.0f}]")

LOADING HISTORICAL FULL-SEASON DATA (2016-2024)

Loading raw hitter data...

Loading raw pitcher data...

Loaded 5760 hitter seasons (raw)
Loaded 7237 pitcher seasons (raw)

Processing hitter data through pipeline...
  Pipeline adds: Age (from BP_Data), features, WAR_per_600


17:25:10 - new_pipeline.common.transformers.filters - INFO - PAFilter: Removed 1524 hitters with < 75 PA (full season)
17:25:10 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Loaded Age for 2088 hitters
17:25:10 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Added Age column (range: 20-45)
17:25:10 - new_pipeline.common.transformers.hitter_features - INFO - Loading hitter features...
17:25:19 - new_pipeline.common.transformers.hitter_features - INFO - Loaded 11 hitter feature sets (33 total columns)
17:25:19 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Learned replacement values for 28 features
17:25:19 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Imputed 70 missing values
17:25:19 - new_pipeline.common.transformers.validators - WARNING - FeatureValidator found issues:
  - Feature 'K%' range [3.09, 51.25] outside expected [0, 50]
  - Feature 'AVG' range [0.09, 0.38] outside expec


Processing pitcher data through pipeline...
  Pipeline adds: Age (from BP_Data), features, WAR_per_162


17:25:19 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Loaded Age for 2272 pitchers
17:25:19 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Added Age column (range: 19-45)
17:25:19 - new_pipeline.common.transformers.pitcher_features - INFO - Loading pitcher features...


17:25:23 - new_pipeline.common.transformers.pitcher_features - INFO - Loaded 13 pitcher feature sets (38 total columns)
17:25:23 - new_pipeline.common.transformers.pitcher_composite_transformer - INFO - Calculating pitcher composite features...
17:25:23 - new_pipeline.common.transformers.pitcher_composite_transformer - INFO - Added 7 composite features
17:25:23 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Learned replacement values for 41 features
17:25:23 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Imputed 10901 missing values
17:25:23 - new_pipeline.common.transformers.validators - WARNING - FeatureValidator found issues:
  - Feature 'BB%' range [0.00, 50.00] outside expected [0, 25]
  - Feature 'K%' range [0.00, 53.00] outside expected [0, 50]
  - Feature 'ERA' range [0.00, 37.50] outside expected [0, 15]
  - Feature 'GB%' range [0.00, 82.79] outside expected [20, 80]
17:25:23 - new_pipeline.common.transformers.feature_s


Processed 4236 qualified hitters
Processed 5652 qualified pitchers
Years: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Data quality checks:
  Hitters with Age: 4236
  Hitters with WAR_per_600: 4236
  Pitchers with Age: 5652
  Pitchers with WAR_per_162: 5652

  Hitter Age range: [20, 45]
  Pitcher Age range: [19, 45]


In [3]:
# Cell 2.5: Load Injury Data (Auto-discover historical years)

from new_pipeline.common.data_preparation.injury_data_loader import load_injury_data_multiple_years

print("="*70)
print("LOADING INJURY DATA")
print("="*70)

try:
    # Construct absolute path using project_root from Cell 1
    injury_data_dir = project_root / "MLB Player Data" / "FanGraphs_Data" / "injuries"
    
    # Auto-discover all historical years (excludes current season for training)
    injury_data_historical = load_injury_data_multiple_years(
        data_dir=str(injury_data_dir),
        auto_discover=True,
        exclude_current_season=True
    )
    
    print(f"Injury data summary:")
    print(f"  Total records: {len(injury_data_historical)}")
    print(f"  Unique players: {injury_data_historical['MLBAMID'].nunique()}")
    print(f"  Years covered: {sorted(injury_data_historical['Year'].unique())}")
    
    if "status" in injury_data_historical.columns:
        print(f"  Status breakdown: {injury_data_historical['status'].value_counts().to_dict()}")
    
    print("Injury data loaded successfully!")
    
except Exception as e:
    print(f"ERROR: Failed to load injury data")
    print(f"  {type(e).__name__}: {e}")
    raise


LOADING INJURY DATA
Discovered injury data years: [2020, 2021, 2022, 2023, 2024]
Loaded 530 injury records for 2020
  Status breakdown: {'Active roster': 253, '45-Day IL': 114, '10-Day IL': 86, 'Player Pool': 20, 'Released': 3, '7-Day IL': 3, 'Outrighted': 2, 'Opted out of season': 1, 'Designated for assignment': 1}
Loaded 1221 injury records for 2021
  Status breakdown: {'Activated': 958, '60-Day IL': 168, '10-Day IL': 81, 'CV-19 IL': 13}
Loaded 981 injury records for 2022
  Status breakdown: {'Activated': 727, '60-Day IL': 160, '15-Day IL': 56, '10-Day IL': 33, 'CV-19 IL': 3, '7-Day IL': 2}
Loaded 856 injury records for 2023
  Status breakdown: {'Activated': 597, '60-Day IL': 170, '15-Day IL': 58, '10-Day IL': 30, '7-Day IL': 1}
Loaded 775 injury records for 2024
  Status breakdown: {'Activated': 775}

Combined injury data: 4363 total records across 5 years
Injury data summary:
  Total records: 4363
  Unique players: 1657
  Years covered: [np.int64(2020), np.int64(2021), np.int64(202

In [4]:
# Cell 3: Create Multipoint Season Splits

print("="*70)
print("CREATING MULTIPOINT SEASON SPLITS")
print("="*70)

# Create splits at 25%, 50%, 75% of season
# Use PROCESSED data (has Age, WAR_per_600, all features)
split_points = [0.25, 0.5, 0.75]

print(f"\nCreating splits at: {split_points}")
print("Each player-season becomes 3 training samples")
print("  - 25% split: 'At 40 games, predict remaining 122 games WAR'")
print("  - 50% split: 'At 81 games, predict remaining 81 games WAR'")
print("  - 75% split: 'At 121 games, predict remaining 41 games WAR'")

print("\nSplitting hitter data (from PROCESSED - has Age + WAR_per_600)...")
hitter_splits = create_multipoint_splits(
    full_season_df=hitter_processed,  # Use processed data
    split_points=split_points,
    player_type='hitter',
    season_length=162
)

print("\nSplitting pitcher data (from PROCESSED - has Age + WAR_per_162)...")
pitcher_splits = create_multipoint_splits(
    full_season_df=pitcher_processed,  # Use processed data
    split_points=split_points,
    player_type='pitcher',
    season_length=162
)

print(f"\nHitter splits: {len(hitter_splits)} samples (from {len(hitter_processed)} full seasons)")
print(f"Pitcher splits: {len(pitcher_splits)} samples (from {len(pitcher_processed)} full seasons)")

# Verify split structure and Age preservation
print("\nSample split row (hitter):")
sample = hitter_splits.iloc[0]
print(f"  Player: {sample.get('Name', 'N/A')}")
print(f"  Year: {sample['Year']}")
print(f"  Age: {sample.get('Age', 'MISSING')}")
print(f"  Split point: {sample['split_point']}")
print(f"  Season completion: {sample['season_completion_pct']:.1%}")
print(f"  Games played: {sample['games_played']:.0f}")
print(f"  Current PA: {sample['current_PA']:.0f}")
print(f"  Remaining PA: {sample['remaining_PA']:.0f}")
print(f"  Remaining WAR (target): {sample['remaining_WAR']:.2f}")

# Verify Age is preserved
print(f"\nAge preserved in splits:")
print(f"  Hitters: {'Age' in hitter_splits.columns} (range: {hitter_splits['Age'].min():.0f}-{hitter_splits['Age'].max():.0f})")
print(f"  Pitchers: {'Age' in pitcher_splits.columns} (range: {pitcher_splits['Age'].min():.0f}-{pitcher_splits['Age'].max():.0f})")

CREATING MULTIPOINT SEASON SPLITS

Creating splits at: [0.25, 0.5, 0.75]
Each player-season becomes 3 training samples
  - 25% split: 'At 40 games, predict remaining 122 games WAR'
  - 50% split: 'At 81 games, predict remaining 81 games WAR'
  - 75% split: 'At 121 games, predict remaining 41 games WAR'

Splitting hitter data (from PROCESSED - has Age + WAR_per_600)...

Splitting pitcher data (from PROCESSED - has Age + WAR_per_162)...

Hitter splits: 12702 samples (from 4236 full seasons)
Pitcher splits: 10503 samples (from 5652 full seasons)

Sample split row (hitter):
  Player: David Ortiz
  Year: 2016
  Age: 40
  Split point: 0.25
  Season completion: 25.0%
  Games played: 38
  Current PA: 156
  Remaining PA: 470
  Remaining WAR (target): 3.42

Age preserved in splits:
  Hitters: True (range: 20-45)
  Pitchers: True (range: 19-45)


In [5]:
# Cell 4: Build ROS Features

print("="*70)
print("BUILDING ROS FEATURES")
print("="*70)

# Build complete ROS feature sets (elite detection, age curves, baselines, etc.)
print("\nInitializing feature builders...")
hitter_builder = ROSFeatureBuilder(player_type='hitter')
pitcher_builder = ROSFeatureBuilder(player_type='pitcher')

print("\nBuilding hitter ROS features...")
print("  current_season_df: splits (has Age, WAR_per_600, all stats)")
print("  historical_df: processed (has Age, WAR_per_600 for lookups)")
hitter_with_features = hitter_builder.build_features_batch(
    current_season_df=hitter_splits,  # Has Age + WAR_per_600
    historical_df=hitter_processed  # Has Age + WAR_per_600
)

print("\nBuilding pitcher ROS features...")
pitcher_with_features = pitcher_builder.build_features_batch(
    current_season_df=pitcher_splits,  # Has Age + WAR_per_162
    historical_df=pitcher_processed  # Has Age + WAR_per_162
)

print(f"\nFeature building complete!")
print(f"  Hitter samples with features: {len(hitter_with_features)}")
print(f"  Pitcher samples with features: {len(pitcher_with_features)}")

# Verify feature columns
hitter_feature_count = len([col for col in hitter_with_features.columns if col in ROS_HITTER_FEATURES])
pitcher_feature_count = len([col for col in pitcher_with_features.columns if col in ROS_PITCHER_FEATURES])

print(f"\nFeature availability check:")
print(f"  Hitter features present: {hitter_feature_count}/{len(ROS_HITTER_FEATURES)} ({hitter_feature_count/len(ROS_HITTER_FEATURES)*100:.1f}%)")
print(f"  Pitcher features present: {pitcher_feature_count}/{len(ROS_PITCHER_FEATURES)} ({pitcher_feature_count/len(ROS_PITCHER_FEATURES)*100:.1f}%)")

# Stop if too many features missing
if hitter_feature_count < len(ROS_HITTER_FEATURES) * 0.8:
    raise ValueError(f"Too many hitter features missing: {hitter_feature_count}/{len(ROS_HITTER_FEATURES)}")
if pitcher_feature_count < len(ROS_PITCHER_FEATURES) * 0.8:
    raise ValueError(f"Too many pitcher features missing: {pitcher_feature_count}/{len(ROS_PITCHER_FEATURES)}")

print("\n" + "="*70)

BUILDING ROS FEATURES

Initializing feature builders...

Building hitter ROS features...
  current_season_df: splits (has Age, WAR_per_600, all stats)
  historical_df: processed (has Age, WAR_per_600 for lookups)

Building pitcher ROS features...

Feature building complete!
  Hitter samples with features: 12702
  Pitcher samples with features: 10503

Feature availability check:
  Hitter features present: 66/66 (100.0%)
  Pitcher features present: 52/52 (100.0%)



In [6]:
# Cell 5: Prepare Training Data

print("="*70)
print("PREPARING TRAINING DATA")
print("="*70)

# Use feature-enriched DataFrames from Cell 4
print("\nPreparing hitter training data...")
hitter_train_df, X_hitter, y_hitter = prepare_ros_training_data(
    multipoint_df=hitter_with_features,  # From Cell 4 (has all features)
    feature_columns=ROS_HITTER_FEATURES,
    target_column='remaining_WAR'
)

print("\nPreparing pitcher training data...")
pitcher_train_df, X_pitcher, y_pitcher = prepare_ros_training_data(
    multipoint_df=pitcher_with_features,  # From Cell 4 (has all features)
    feature_columns=ROS_PITCHER_FEATURES,
    target_column='remaining_WAR'
)

print(f"\n{'='*70}")
print("TRAINING DATA SUMMARY")
print(f"{'='*70}")

print(f"\nHitters:")
print(f"  Samples: {len(X_hitter)}")
print(f"  Features: {X_hitter.shape[1]} (expected: {len(ROS_HITTER_FEATURES)})")
print(f"  Target (remaining WAR):")
print(f"    Mean: {y_hitter.mean():.2f}")
print(f"    Std: {y_hitter.std():.2f}")
print(f"    Range: [{y_hitter.min():.2f}, {y_hitter.max():.2f}]")

print(f"\nPitchers:")
print(f"  Samples: {len(X_pitcher)}")
print(f"  Features: {X_pitcher.shape[1]} (expected: {len(ROS_PITCHER_FEATURES)})")
print(f"  Target (remaining WAR):")
print(f"    Mean: {y_pitcher.mean():.2f}")
print(f"    Std: {y_pitcher.std():.2f}")
print(f"    Range: [{y_pitcher.min():.2f}, {y_pitcher.max():.2f}]")

# Verify feature counts match
assert X_hitter.shape[1] == len(ROS_HITTER_FEATURES), f"Hitter feature mismatch: {X_hitter.shape[1]} != {len(ROS_HITTER_FEATURES)}"
assert X_pitcher.shape[1] == len(ROS_PITCHER_FEATURES), f"Pitcher feature mismatch: {X_pitcher.shape[1]} != {len(ROS_PITCHER_FEATURES)}"

print(f"\n{'='*70}")
print("READY FOR TRAINING")
print(f"{'='*70}")

PREPARING TRAINING DATA

Preparing hitter training data...

Preparing pitcher training data...

TRAINING DATA SUMMARY

Hitters:
  Samples: 12702
  Features: 66 (expected: 66)
  Target (remaining WAR):
    Mean: 0.58
    Std: 0.96
    Range: [-1.91, 8.50]

Pitchers:
  Samples: 10503
  Features: 52 (expected: 52)
  Target (remaining WAR):
    Mean: 0.48
    Std: 0.73
    Range: [-1.03, 6.78]

READY FOR TRAINING


In [7]:
# Cell 6: Train Hitter ROS Ensemble

print("="*70)
print("TRAINING HITTER ROS ENSEMBLE")
print("="*70)

# Initialize ensemble with validated weights
print("\nInitializing HitterROSEnsemble...")
hitter_ros = HitterROSEnsemble(
    weights=[0.5, 0.4, 0.1],  # DirectROSForecaster, DartsTemporalEnsemble, Baseline
    feature_columns=ROS_HITTER_FEATURES,
    target_column='remaining_WAR'
)

# Train on multipoint historical data
print("\nFitting ensemble on historical data (2016-2024)...")
print("This may take several minutes...")

hitter_ros.fit(
    historical_df=hitter_train_df,
    feature_columns=ROS_HITTER_FEATURES,
    target_column='remaining_WAR'
)

print("\n" + "="*70)
print("HITTER ROS ENSEMBLE TRAINING COMPLETE")
print("="*70)

TRAINING HITTER ROS ENSEMBLE

Initializing HitterROSEnsemble...

Fitting ensemble on historical data (2016-2024)...
This may take several minutes...
Fitting HitterROSEnsemble on 12702 samples...
  Converting to sktime format (DirectROSForecaster)...
  Fitting DirectROSForecaster (11835 samples after validation)...
  Converting to Darts format (Temporal ensemble)...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


  Fitting DartsTemporalEnsemble (509 player series)...


c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\pytorch_lightning\core\module.py:512: You called `self.log('train_Bias', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\pytorch_lightning\core\module.py:512: You called `self.log('train_Elite_Bias', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\pytorch_lightning\core\module.py:512: You called `self.log('train_Elite_MAE', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\pytorch_lightning\core\module.py:512: You called `self.log('train_MAE', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=A

    Note: AutoARIMA skipped for 509/509 players (<10 years data)
  Preparing baseline training data...
  Fitting baseline MultiQuantileHistGB (12702 samples)...
Ensemble fitting complete.

HITTER ROS ENSEMBLE TRAINING COMPLETE


In [8]:
# Cell 7: Train Pitcher ROS Ensemble

print("="*70)
print("TRAINING PITCHER ROS ENSEMBLE")
print("="*70)

# Initialize ensemble with validated weights
print("\nInitializing PitcherROSEnsemble...")
pitcher_ros = PitcherROSEnsemble(
    weights=[0.5, 0.4, 0.1],  # DirectROSForecaster, DartsTemporalEnsemble, Baseline
    feature_columns=ROS_PITCHER_FEATURES,
    target_column='remaining_WAR'
)

# Train on multipoint historical data
print("\nFitting ensemble on historical data (2016-2024)...")
print("This may take several minutes...")

pitcher_ros.fit(
    historical_df=pitcher_train_df,
    feature_columns=ROS_PITCHER_FEATURES,
    target_column='remaining_WAR'
)

print("\n" + "="*70)
print("PITCHER ROS ENSEMBLE TRAINING COMPLETE")
print("="*70)

TRAINING PITCHER ROS ENSEMBLE

Initializing PitcherROSEnsemble...

Fitting ensemble on historical data (2016-2024)...
This may take several minutes...
Fitting PitcherROSEnsemble on 10503 samples...
  Converting to sktime format (DirectROSForecaster)...
  Fitting DirectROSForecaster (9321 samples after validation)...
  Converting to Darts format (Temporal ensemble)...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


  Fitting DartsTemporalEnsemble (387 player series)...


c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\pytorch_lightning\core\module.py:512: You called `self.log('train_Bias', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\pytorch_lightning\core\module.py:512: You called `self.log('train_Elite_Bias', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\pytorch_lightning\core\module.py:512: You called `self.log('train_Elite_MAE', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\pytorch_lightning\core\module.py:512: You called `self.log('train_MAE', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=A

    Note: AutoARIMA skipped for 387/387 players (<10 years data)
  Preparing baseline training data...
  Fitting baseline MultiQuantileHistGB (10503 samples)...
Ensemble fitting complete.

PITCHER ROS ENSEMBLE TRAINING COMPLETE


In [9]:
# Cell 8: Validate Training

print("="*70)
print("VALIDATING TRAINED MODELS")
print("="*70)

# Test hitter predictions on small sample
print("\nTesting hitter ROS predictions...")
sample_size = 10
# Use DataFrame slicing and pass historical data for cascading fallback
hitter_sample_pred = hitter_ros.predict(
    hitter_train_df.iloc[:sample_size],  # DataFrame with playerid column
    hitter_train_df  # Historical data for tier-based predictions
)
print(f"  Sample predictions: {hitter_sample_pred}")
print(f"  Sample actuals:     {y_hitter[:sample_size]}")
print(f"  Prediction shape: {hitter_sample_pred.shape}")

# Test pitcher predictions
print("\nTesting pitcher ROS predictions...")
pitcher_sample_pred = pitcher_ros.predict(
    pitcher_train_df.iloc[:sample_size],  # DataFrame with playerid column
    pitcher_train_df  # Historical data for tier-based predictions
)
print(f"  Sample predictions: {pitcher_sample_pred}")
print(f"  Sample actuals:     {y_pitcher[:sample_size]}")
print(f"  Prediction shape: {pitcher_sample_pred.shape}")

# Verify predictions are reasonable
assert len(hitter_sample_pred) == sample_size, "Hitter prediction failed"
assert len(pitcher_sample_pred) == sample_size, "Pitcher prediction failed"
assert not np.isnan(hitter_sample_pred).any(), "Hitter predictions contain NaN"
assert not np.isnan(pitcher_sample_pred).any(), "Pitcher predictions contain NaN"

print("\n" + "="*70)
print("VALIDATION PASSED")
print("="*70)

VALIDATING TRAINED MODELS

Testing hitter ROS predictions...
  Player tiers: Tier1=0, Tier2=4, Tier3=6
  Sample predictions: [ 3.41267284  2.28178011  1.13373533 -0.7959371  -0.54934516 -0.30919054
  3.73355108  2.54027687  1.29273879  2.0164006 ]
  Sample actuals:     [ 3.42404837  2.28269891  1.14134946 -0.91947216 -0.61298144 -0.30649072
  3.86893033  2.57928689  1.28964344  1.9195067 ]
  Prediction shape: (10,)

Testing pitcher ROS predictions...
  Player tiers: Tier1=0, Tier2=9, Tier3=1
  Sample predictions: [ 1.8910973   1.25374318  0.65342523  0.23840885  0.1515712   0.09717477
  0.2861267   0.19351997  0.13226602 -0.05218834]
  Sample actuals:     [ 1.82988485  1.21992323  0.60996162  0.27455395  0.18303597  0.09151798
  0.32393507  0.21595671  0.10797836 -0.00317843]
  Prediction shape: (10,)

VALIDATION PASSED


In [10]:
# Cell 9: Save Trained Models and Historical Split Data

print("="*70)
print("SAVING TRAINED MODELS AND SPLIT DATA")
print("="*70)

# Create models directory
models_dir = project_root / 'models'
models_dir.mkdir(exist_ok=True)

# Save hitter ROS model
hitter_path = models_dir / 'hitter_ros_2025.pkl'
joblib.dump(hitter_ros, hitter_path)
print(f"\nSaved: {hitter_path}")

# Verify file size (trained models should be > 100KB)
hitter_size_mb = hitter_path.stat().st_size / 1024 / 1024
print(f"  File size: {hitter_size_mb:.1f} MB")
assert hitter_size_mb > 0.1, f"Hitter model too small ({hitter_size_mb:.1f} MB) - likely not trained"

# Save hitter Darts temporal models separately (they don't serialize with joblib)
if hitter_ros.temporal_model_fitted:
    print("\nSaving hitter Darts temporal models...")
    hitter_tcn_path = models_dir / 'hitter_ros_tcn_2025.pt'
    hitter_tsmixer_path = models_dir / 'hitter_ros_tsmixer_2025.pt'
    
    hitter_ros.temporal_model.tcn.save(str(hitter_tcn_path))
    hitter_ros.temporal_model.tsmixer.save(str(hitter_tsmixer_path))
    
    print(f"  Saved TCN: {hitter_tcn_path.name}")
    print(f"  Saved TSMixer: {hitter_tsmixer_path.name}")
else:
    print("\n  Note: Hitter temporal model not fitted, skipping Darts model save")

# Save pitcher ROS model
pitcher_path = models_dir / 'pitcher_ros_2025.pkl'
joblib.dump(pitcher_ros, pitcher_path)
print(f"\nSaved: {pitcher_path}")

# Verify file size
pitcher_size_mb = pitcher_path.stat().st_size / 1024 / 1024
print(f"  File size: {pitcher_size_mb:.1f} MB")
assert pitcher_size_mb > 0.1, f"Pitcher model too small ({pitcher_size_mb:.1f} MB) - likely not trained"

# Save pitcher Darts temporal models separately (they don't serialize with joblib)
if pitcher_ros.temporal_model_fitted:
    print("\nSaving pitcher Darts temporal models...")
    pitcher_tcn_path = models_dir / 'pitcher_ros_tcn_2025.pt'
    pitcher_tsmixer_path = models_dir / 'pitcher_ros_tsmixer_2025.pt'
    
    pitcher_ros.temporal_model.tcn.save(str(pitcher_tcn_path))
    pitcher_ros.temporal_model.tsmixer.save(str(pitcher_tsmixer_path))
    
    print(f"  Saved TCN: {pitcher_tcn_path.name}")
    print(f"  Saved TSMixer: {pitcher_tsmixer_path.name}")
else:
    print("\n  Note: Pitcher temporal model not fitted, skipping Darts model save")

print("\n" + "="*70)
print("SAVING HISTORICAL SPLIT DATA (for inference)")
print("="*70)

# Save historical split data (with remaining_WAR) for inference
# This data will be used to provide historical context during ROS predictions
hitter_splits_path = models_dir / 'hitter_splits_2016_2024.pkl'
joblib.dump(hitter_with_features, hitter_splits_path)
print(f"\nSaved: {hitter_splits_path}")

hitter_splits_size_mb = hitter_splits_path.stat().st_size / 1024 / 1024
print(f"  File size: {hitter_splits_size_mb:.1f} MB")
print(f"  Rows: {len(hitter_with_features)}")
print(f"  Columns: {len(hitter_with_features.columns)}")
print(f"  Split points: {sorted(hitter_with_features['split_point'].unique())}")

pitcher_splits_path = models_dir / 'pitcher_splits_2016_2024.pkl'
joblib.dump(pitcher_with_features, pitcher_splits_path)
print(f"\nSaved: {pitcher_splits_path}")

pitcher_splits_size_mb = pitcher_splits_path.stat().st_size / 1024 / 1024
print(f"  File size: {pitcher_splits_size_mb:.1f} MB")
print(f"  Rows: {len(pitcher_with_features)}")
print(f"  Columns: {len(pitcher_with_features.columns)}")
print(f"  Split points: {sorted(pitcher_with_features['split_point'].unique())}")

print("\n" + "="*70)
print("MODELS AND SPLIT DATA SAVED SUCCESSFULLY")
print("="*70)
print("\nSaved files:")
print(f"  - {hitter_path.name} (trained model)")
if hitter_ros.temporal_model_fitted:
    print(f"  - hitter_ros_tcn_2025.pt (Darts TCN model)")
    print(f"  - hitter_ros_tsmixer_2025.pt (Darts TSMixer model)")
print(f"  - {hitter_splits_path.name} (historical splits with remaining_WAR)")
print(f"  - {pitcher_path.name} (trained model)")
if pitcher_ros.temporal_model_fitted:
    print(f"  - pitcher_ros_tcn_2025.pt (Darts TCN model)")
    print(f"  - pitcher_ros_tsmixer_2025.pt (Darts TSMixer model)")
print(f"  - {pitcher_splits_path.name} (historical splits with remaining_WAR)")
print("\nNext: Load these in oWAR_overview.ipynb for inference")

SAVING TRAINED MODELS AND SPLIT DATA

Saved: c:\Users\nairs\Documents\GithubProjects\oWAR\models\hitter_ros_2025.pkl
  File size: 70.1 MB

Saving hitter Darts temporal models...
  Saved TCN: hitter_ros_tcn_2025.pt
  Saved TSMixer: hitter_ros_tsmixer_2025.pt

Saved: c:\Users\nairs\Documents\GithubProjects\oWAR\models\pitcher_ros_2025.pkl
  File size: 51.2 MB

Saving pitcher Darts temporal models...
  Saved TCN: pitcher_ros_tcn_2025.pt
  Saved TSMixer: pitcher_ros_tsmixer_2025.pt

SAVING HISTORICAL SPLIT DATA (for inference)

Saved: c:\Users\nairs\Documents\GithubProjects\oWAR\models\hitter_splits_2016_2024.pkl
  File size: 14.2 MB
  Rows: 12702
  Columns: 147
  Split points: [np.float64(0.25), np.float64(0.5), np.float64(0.75)]

Saved: c:\Users\nairs\Documents\GithubProjects\oWAR\models\pitcher_splits_2016_2024.pkl
  File size: 10.0 MB
  Rows: 10503
  Columns: 126
  Split points: [np.float64(0.25), np.float64(0.5), np.float64(0.75)]

MODELS AND SPLIT DATA SAVED SUCCESSFULLY

Saved files

In [11]:
# Cell 9.5: Pitcher ROS Feature Building & Prediction

# Suppress PyTorch Lightning verbosity for cleaner output
import os
import warnings
import logging

os.environ['PYTORCH_LIGHTNING_VERBOSITY'] = '0'
warnings.filterwarnings('ignore', category=UserWarning, module='pytorch_lightning')
warnings.filterwarnings('ignore', category=FutureWarning, module='pytorch_lightning')
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning.utilities.rank_zero").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning.accelerators.cuda").setLevel(logging.ERROR)

print("="*90)
print("PITCHER ROS FEATURE BUILDING & PREDICTION")
print("="*90)
print()

# Load and process current season data
from new_pipeline.notebooks.shared.pipeline_runner import load_current_season_data, run_data_pipeline
from new_pipeline.common.data_preparation.clean_multi_team_data import clean_multi_team_players

# Reload modules to ensure we have the latest version
import importlib
import new_pipeline.common.projections.usage_projections
import new_pipeline.common.projections.ros_projections
importlib.reload(new_pipeline.common.projections.usage_projections)
importlib.reload(new_pipeline.common.projections.ros_projections)

from new_pipeline.common.projections.usage_projections import (
    get_team_games_from_data,
    calculate_pitcher_remaining_ip,
    classify_pitcher_role  # NEW: Import centralized classification
)
from new_pipeline.common.projections.ros_projections import format_pitcher_ros_display

pitcher_2025_raw = load_current_season_data('pitcher', year=2025)

# Clean multi-team players
pitcher_2025_raw = clean_multi_team_players(pitcher_2025_raw, year=2025, player_type="pitcher")
print(f"Cleaned pitchers - Multi-team players: {sum(pitcher_2025_raw['Team'].str.contains(',', na=False))}")
pitcher_2025_processed = run_data_pipeline(pitcher_2025_raw, player_type='pitcher')

# Load hitter data to get accurate team games played
hitter_2025_raw = load_current_season_data('hitter', year=2025)

# Clean multi-team players
hitter_2025_raw = clean_multi_team_players(hitter_2025_raw, year=2025, player_type="hitter")
print(f"Cleaned hitters - Multi-team players: {sum(hitter_2025_raw['Team'].str.contains(',', na=False))}")
hitter_2025_processed = run_data_pipeline(hitter_2025_raw, player_type='hitter')

# Get team-specific games played using utility function
team_games_dict, league_median_games = get_team_games_from_data(hitter_2025_processed)

# Calculate season progression
season_pct = league_median_games / 162
remaining_team_games = 162 - league_median_games

print(f"League median games: {league_median_games:.0f}, Remaining: {remaining_team_games:.0f}, Season: {season_pct:.1%}")
print(f"(Individual projections use team-specific games + multi-team handling)")
print()

# ============================================================================
# CLASSIFY ALL PITCHERS USING CENTRALIZED FUNCTION
# ============================================================================
print("Classifying all pitchers using classify_pitcher_role()...")

# Apply classification to ALL pitchers
pitcher_2025_processed['role'] = pitcher_2025_processed.apply(
    lambda row: classify_pitcher_role(
        games_started=row['GS'],
        games_pitched=row['G'],
        innings_pitched=row['IP']
    ),
    axis=1
)

# Segment into three groups
starters_all = pitcher_2025_processed[pitcher_2025_processed['role'] == 'starter'].copy()
swing_all = pitcher_2025_processed[pitcher_2025_processed['role'] == 'swing'].copy()
relievers_all = pitcher_2025_processed[pitcher_2025_processed['role'] == 'reliever'].copy()

print(f"  Starters: {len(starters_all)}")
print(f"  Swing: {len(swing_all)}")
print(f"  Relievers: {len(relievers_all)}")
print()

# ============================================================================
# STARTERS (ALL qualified, then sort by ROS_WAR)
# ============================================================================
print("Building features for ALL STARTERS...")

# Build ROS features for ALL starters
pitcher_ros_features_starters = pitcher_builder.build_features_batch(
    current_season_df=starters_all,
    historical_df=pitcher_processed,
    injury_df=injury_data_historical
)

# Run complete ROS prediction workflow
from new_pipeline.common.projections.ros_projections import run_ros_predictions

results_starters = run_ros_predictions(
    ensemble=pitcher_ros,
    current_df=pitcher_ros_features_starters,
    historical_df=pitcher_with_features,
    player_type='pitcher',
    role='starter',
    season_pct=season_pct,
    blend_ratio=0.70,
    team_games_dict=team_games_dict,
    league_median_games=league_median_games,
    use_percentile_tiers=True
)

# Unpack results
ros_predictions_starters = results_starters['predictions']
tier_labels_starters_all = results_starters['tiers']
projected_remaining_ip_all = results_starters['usage_projected']
ros_display_starters_all = results_starters['display']

# Get thresholds for display
# Get percentile targets and actual cutoffs
from new_pipeline.models.ros.tier_thresholds import get_tier_percentiles
elite_pct, good_pct = get_tier_percentiles('starter')
elite_cutoff = np.percentile(ros_predictions_starters['mean'], 100 * (1 - elite_pct))
good_cutoff = np.percentile(ros_predictions_starters['mean'], 100 * (1 - good_pct))

print(f"Tier Classification (Percentile-Based):")
print(f"  Elite: Top {elite_pct*100:.1f}% (~{int(len(starters_all) * elite_pct)} of {len(starters_all)} players)")
print(f"    -> Cutoff this run: {elite_cutoff:.2f} ROS WAR")
print(f"  Good: Top {good_pct*100:.1f}% (~{int(len(starters_all) * good_pct)} of {len(starters_all)} players)")
print(f"    -> Cutoff this run: {good_cutoff:.2f} ROS WAR")
print()


# Extract additional usage stats for display
starter_team_games_all = starters_all['Team'].map(team_games_dict).fillna(league_median_games).values
ip_per_start_all = starters_all['IP'].values / np.maximum(starters_all['GS'].values, 1)


# Sort by ROS_WAR (descending) and take top 20
starter_ros_ranking = np.argsort(ros_display_starters_all['ros_war'])[::-1]
top_20_starter_idx = starter_ros_ranking[:20]

# Apply sorting to create top 20 arrays
starters = starters_all.iloc[top_20_starter_idx].reset_index(drop=True)
tier_labels_starters = tier_labels_starters_all[top_20_starter_idx]
projected_remaining_ip = projected_remaining_ip_all[top_20_starter_idx]
starter_team_games = starter_team_games_all[top_20_starter_idx]
ip_per_start = ip_per_start_all[top_20_starter_idx]
ros_display_starters = {
    'ros_war': ros_display_starters_all['ros_war'][top_20_starter_idx],
    'ros_rate': ros_display_starters_all['ros_rate'][top_20_starter_idx],
    'ros_q50': ros_display_starters_all['ros_q50'][top_20_starter_idx],
    'ros_q90': ros_display_starters_all['ros_q90'][top_20_starter_idx]
}

print(f"  Predicted all {len(starters_all)} starters, selected top 20 by ROS_WAR")

# ============================================================================
# SWING PITCHERS (ALL qualified, then sort by ROS_WAR)
# ============================================================================
print("Building features for ALL SWING PITCHERS...")

# Build ROS features for ALL swing pitchers
pitcher_ros_features_swing = pitcher_builder.build_features_batch(
    current_season_df=swing_all,
    historical_df=pitcher_processed,
    injury_df=injury_data_historical
)

# Run complete ROS prediction workflow
from new_pipeline.common.projections.ros_projections import run_ros_predictions

results_swing = run_ros_predictions(
    ensemble=pitcher_ros,
    current_df=pitcher_ros_features_swing,
    historical_df=pitcher_with_features,
    player_type='pitcher',
    role='swing',
    season_pct=season_pct,
    blend_ratio=0.70,
    team_games_dict=team_games_dict,
    league_median_games=league_median_games,
    use_percentile_tiers=True
)

# Unpack results
ros_predictions_swing = results_swing['predictions']
tier_labels_swing_all = results_swing['tiers']
projected_remaining_ip_swing_all = results_swing['usage_projected']
ros_display_swing_all = results_swing['display']

# Get thresholds for display
# Get percentile targets and actual cutoffs
from new_pipeline.models.ros.tier_thresholds import get_tier_percentiles
elite_pct, good_pct = get_tier_percentiles('swing')
elite_cutoff = np.percentile(ros_predictions_swing['mean'], 100 * (1 - elite_pct))
good_cutoff = np.percentile(ros_predictions_swing['mean'], 100 * (1 - good_pct))

print(f"Tier Classification (Percentile-Based):")
print(f"  Elite: Top {elite_pct*100:.1f}% (~{int(len(swing_all) * elite_pct)} of {len(swing_all)} players)")
print(f"    -> Cutoff this run: {elite_cutoff:.2f} ROS WAR")
print(f"  Good: Top {good_pct*100:.1f}% (~{int(len(swing_all) * good_pct)} of {len(swing_all)} players)")
print(f"    -> Cutoff this run: {good_cutoff:.2f} ROS WAR")
print()


# Extract additional usage stats for display
swing_team_games_all = swing_all['Team'].map(team_games_dict).fillna(league_median_games).values
ip_per_appearance_swing_all = swing_all['IP'].values / np.maximum(swing_all['G'].values, 1)
appearance_rate_swing_all = swing_all['G'].values / swing_team_games_all


# Sort by ROS_WAR (descending) and take top 20
swing_ros_ranking = np.argsort(ros_display_swing_all['ros_war'])[::-1]
top_20_swing_idx = swing_ros_ranking[:20]

# Apply sorting to create top 20 arrays
swing_pitchers = swing_all.iloc[top_20_swing_idx].reset_index(drop=True)
tier_labels_swing = tier_labels_swing_all[top_20_swing_idx]
projected_remaining_ip_swing = projected_remaining_ip_swing_all[top_20_swing_idx]
swing_team_games = swing_team_games_all[top_20_swing_idx]
ip_per_appearance_swing = ip_per_appearance_swing_all[top_20_swing_idx]
appearance_rate_swing = appearance_rate_swing_all[top_20_swing_idx]
ros_display_swing = {
    'ros_war': ros_display_swing_all['ros_war'][top_20_swing_idx],
    'ros_rate': ros_display_swing_all['ros_rate'][top_20_swing_idx],
    'ros_q50': ros_display_swing_all['ros_q50'][top_20_swing_idx],
    'ros_q90': ros_display_swing_all['ros_q90'][top_20_swing_idx]
}

print(f"  Predicted all {len(swing_all)} swing pitchers, selected top 20 by ROS_WAR")

# ============================================================================
# RELIEVERS (ALL qualified, then sort by ROS_WAR)
# ============================================================================
print("Building features for ALL RELIEVERS...")

# Build ROS features for ALL relievers
pitcher_ros_features_relievers = pitcher_builder.build_features_batch(
    current_season_df=relievers_all,
    historical_df=pitcher_processed,
    injury_df=injury_data_historical
)

# Run complete ROS prediction workflow
from new_pipeline.common.projections.ros_projections import run_ros_predictions

results_relievers = run_ros_predictions(
    ensemble=pitcher_ros,
    current_df=pitcher_ros_features_relievers,
    historical_df=pitcher_with_features,
    player_type='pitcher',
    role='reliever',
    season_pct=season_pct,
    blend_ratio=0.70,
    team_games_dict=team_games_dict,
    league_median_games=league_median_games,
    use_percentile_tiers=True
)

# Unpack results
ros_predictions_relievers = results_relievers['predictions']
tier_labels_relievers_all = results_relievers['tiers']
projected_remaining_ip_rel_all = results_relievers['usage_projected']
ros_display_relievers_all = results_relievers['display']

# Get thresholds for display
# Get percentile targets and actual cutoffs
from new_pipeline.models.ros.tier_thresholds import get_tier_percentiles
elite_pct, good_pct = get_tier_percentiles('reliever')
elite_cutoff = np.percentile(ros_predictions_relievers['mean'], 100 * (1 - elite_pct))
good_cutoff = np.percentile(ros_predictions_relievers['mean'], 100 * (1 - good_pct))

print(f"Tier Classification (Percentile-Based):")
print(f"  Elite: Top {elite_pct*100:.1f}% (~{int(len(relievers_all) * elite_pct)} of {len(relievers_all)} players)")
print(f"    -> Cutoff this run: {elite_cutoff:.2f} ROS WAR")
print(f"  Good: Top {good_pct*100:.1f}% (~{int(len(relievers_all) * good_pct)} of {len(relievers_all)} players)")
print(f"    -> Cutoff this run: {good_cutoff:.2f} ROS WAR")
print()


# Extract additional usage stats for display
reliever_team_games_all = relievers_all['Team'].map(team_games_dict).fillna(league_median_games).values
ip_per_appearance_all = relievers_all['IP'].values / np.maximum(relievers_all['G'].values, 1)
appearance_rate_all = relievers_all['G'].values / reliever_team_games_all


# Sort by ROS_WAR (descending) and take top 20
reliever_ros_ranking = np.argsort(ros_display_relievers_all['ros_war'])[::-1]
top_20_reliever_idx = reliever_ros_ranking[:20]

# Apply sorting to create top 20 arrays
relievers = relievers_all.iloc[top_20_reliever_idx].reset_index(drop=True)
tier_labels_relievers = tier_labels_relievers_all[top_20_reliever_idx]
projected_remaining_ip_rel = projected_remaining_ip_rel_all[top_20_reliever_idx]
reliever_team_games = reliever_team_games_all[top_20_reliever_idx]
ip_per_appearance = ip_per_appearance_all[top_20_reliever_idx]
appearance_rate = appearance_rate_all[top_20_reliever_idx]
ros_display_relievers = {
    'ros_war': ros_display_relievers_all['ros_war'][top_20_reliever_idx],
    'ros_rate': ros_display_relievers_all['ros_rate'][top_20_reliever_idx],
    'ros_q50': ros_display_relievers_all['ros_q50'][top_20_reliever_idx],
    'ros_q90': ros_display_relievers_all['ros_q90'][top_20_reliever_idx]
}

print(f"  Predicted all {len(relievers_all)} relievers, selected top 20 by ROS_WAR")
print()
print("Feature building & prediction complete!")
print("="*90)

PITCHER ROS FEATURE BUILDING & PREDICTION

Loading partial season data: fangraphs_pitchers_2025_firsthalf.csv
Found 33 multi-team pitcher(s) to clean
Gathering player lookup table. This may take a moment.
Using cached stint data for player 16943 (2025)
  ✓ Sean Newcomb: ATH, BOS (current: BOS)
Using cached stint data for player 17735 (2025)
  ✓ Tyler Alexander: CHW, MIL (current: MIL)
Using cached stint data for player 19618 (2025)
  ✓ Jordan Hicks: BOS, SFG (current: SFG)
Using cached stint data for player 12760 (2025)
  ✓ Rafael Montero: ATL, HOU, DET (current: DET)
Using cached stint data for player 6984 (2025)
  ✓ Luis García: LAD, LAA, WSN (current: WSN)
Using cached stint data for player 19804 (2025)
  ✓ Bryan Baker: BAL, TBR (current: TBR)
Using cached stint data for player 11804 (2025)
  ✓ Héctor Neris: LAA, HOU, ATL (current: ATL)
Using cached stint data for player 19459 (2025)
  ✓ Jorge Alcala: MIN, BOS, STL (current: STL)
Using cached stint data for player 22713 (2025)
  ✓ C

17:43:48 - new_pipeline.common.transformers.filters - INFO - IPFilter: Removed 157 pitchers (position players / insufficient sample, partial season)
17:43:48 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Loaded Age for 873 pitchers
17:43:48 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Added Age column (range: 20-42)
17:43:48 - new_pipeline.common.transformers.pitcher_features - INFO - Loading pitcher features...


Using cached stint data for player 16610 (2025)
  ✓ J.P. Feyereisen: ARI, LAD (current: LAD)
Using cached stint data for player 17169 (2025)
  ✓ José Castillo: NYM, BAL, ARI, SEA (current: SEA)
Using cached stint data for player 19337 (2025)
  ✓ Richard Lovelady: TOR, NYM (current: NYM)
Using cached stint data for player 22872 (2025)
  ✓ Yoendrys Gómez: CHW, NYY, LAD (current: LAD)
Using cached stint data for player 13652 (2025)
  ✓ Tayler Scott: HOU, ARI (current: ARI)
Using cached stint data for player 17490 (2025)
  ✓ Génesis Cabrera: MIN, PIT, CHC, NYM (current: NYM)
Using cached stint data for player 20185 (2025)
  ✓ Zach Pop: SEA, NYM (current: NYM)
Using cached stint data for player 16427 (2025)
  ✓ Scott Blewett: BAL, ATL, MIN (current: MIN)
Using cached stint data for player 10591 (2025)
  ✓ Scott Alexander: COL, SFG (current: SFG)
Cleaned pitchers - Multi-team players: 29


17:43:49 - new_pipeline.common.transformers.pitcher_features - INFO - Loaded 13 pitcher feature sets (40 total columns)
17:43:49 - new_pipeline.common.transformers.pitcher_composite_transformer - INFO - Calculating pitcher composite features...
17:43:49 - new_pipeline.common.transformers.pitcher_composite_transformer - INFO - Added 7 composite features
17:43:49 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Learned replacement values for 41 features
17:43:49 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Imputed 883 missing values
17:43:49 - new_pipeline.common.transformers.validators - WARNING - FeatureValidator found issues:
  - Feature 'BB%' range [0.00, 26.92] outside expected [0, 25]
  - Feature 'ERA' range [0.00, 19.86] outside expected [0, 15]
  - Feature 'GB%' range [11.11, 74.71] outside expected [20, 80]
17:43:49 - new_pipeline.common.transformers.feature_selector - INFO - FeatureSelector: Selected 14 features + 12 met

Loading partial season data: fangraphs_hitters_2025_firsthalf.csv
Found 25 multi-team hitter(s) to clean
Using cached stint data for player 17350 (2025)
  ✓ Rafael Devers: SFG, BOS (current: BOS)
Using cached stint data for player 20572 (2025)
  ✓ Kody Clemens: PHI, MIN (current: MIN)
Using cached stint data for player 15271 (2025)
  ✓ Austin Wynns: ATH, CIN (current: CIN)
Using cached stint data for player 19318 (2025)
  ✓ Matt Thaiss: CHW, TBR (current: TBR)
Using cached stint data for player 19458 (2025)
  ✓ Cooper Hummel: HOU, BAL (current: BAL)
Using cached stint data for player 17735 (2025)
  ✓ Tyler Alexander: CHW, MIL (current: MIL)
Using cached stint data for player 19262 (2025)
  ✓ Garrett Hampson: STL, ARI, CIN (current: CIN)
Using cached stint data for player 13768 (2025)
  ✓ Travis Jankowski: TBR, CHW, NYM (current: NYM)
Using cached stint data for player 16953 (2025)
  ✓ Chadwick Tromp: BAL, ATL (current: ATL)
Using cached stint data for player 12155 (2025)
  ✓ Eddie Rosa

17:44:15 - new_pipeline.common.transformers.filters - INFO - PAFilter: Removed 118 hitters with < 37 PA (partial season)
17:44:15 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Loaded Age for 673 hitters
17:44:15 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Added Age column (range: 21-41)
17:44:15 - new_pipeline.common.transformers.hitter_features - INFO - Loading hitter features...


Using cached stint data for player 13185 (2025)
  ✓ Orlando Arcia: ATL, COL (current: COL)
Using cached stint data for player 24703 (2025)
  ✓ Jonah Bride: MIN, MIA (current: MIA)
Using cached stint data for player 16572 (2025)
  ✓ Connor Joe: CIN, SDP (current: SDP)
Using cached stint data for player 18900 (2025)
  ✓ Leody Taveras: TEX, SEA (current: SEA)
Using cached stint data for player 26197 (2025)
  ✓ Andrew Vaughn: CHW, MIL (current: MIL)
Using cached stint data for player 25040 (2025)
  ✓ Vinny Capra: MIL, CHW (current: CHW)
Using cached stint data for player 13723 (2025)
  ✓ Jacob Stallings: COL, BAL (current: BAL)
Using cached stint data for player 18126 (2025)
  ✓ LaMonte Wade Jr.: SFG, LAA (current: LAA)
Cleaned hitters - Multi-team players: 22


17:44:16 - new_pipeline.common.transformers.hitter_features - INFO - Loaded 11 hitter feature sets (35 total columns)
17:44:16 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Learned replacement values for 28 features
17:44:16 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Imputed 99 missing values
17:44:16 - new_pipeline.common.transformers.validators - WARNING - FeatureValidator found issues:
  - Feature 'AVG' range [0.07, 0.36] outside expected [0.1, 0.4]
  - Feature 'OBP' range [0.12, 0.47] outside expected [0.2, 0.5]
  - Feature 'SLG' range [0.07, 0.74] outside expected [0.2, 0.8]
17:44:16 - new_pipeline.common.transformers.feature_selector - INFO - FeatureSelector: Selected 9 features + 12 metadata columns
17:44:16 - new_pipeline.common.transformers.normalizers - INFO - WARNormalizer: Added 'WAR_per_600' column


League median games: 93, Remaining: 69, Season: 57.4%
(Individual projections use team-specific games + multi-team handling)

Classifying all pitchers using classify_pitcher_role()...
  Starters: 198
  Swing: 66
  Relievers: 333

Building features for ALL STARTERS...
  Player tiers: Tier1=71, Tier2=19, Tier3=108
Tier Classification (Percentile-Based):
  Elite: Top 6.0% (~11 of 198 players)
    -> Cutoff this run: 1.98 ROS WAR
  Good: Top 14.0% (~27 of 198 players)
    -> Cutoff this run: 1.51 ROS WAR

  Predicted all 198 starters, selected top 20 by ROS_WAR
Building features for ALL SWING PITCHERS...
  Player tiers: Tier1=14, Tier2=1, Tier3=51
Tier Classification (Percentile-Based):
  Elite: Top 6.0% (~3 of 66 players)
    -> Cutoff this run: 0.51 ROS WAR
  Good: Top 12.0% (~7 of 66 players)
    -> Cutoff this run: 0.41 ROS WAR

  Predicted all 66 swing pitchers, selected top 20 by ROS_WAR
Building features for ALL RELIEVERS...
  Player tiers: Tier1=104, Tier2=32, Tier3=197
Tier Classi

In [12]:
# Cell 9.6: Hitter ROS Feature Building & Prediction

# Suppress PyTorch Lightning verbosity for cleaner output
import os
import warnings
import logging

os.environ['PYTORCH_LIGHTNING_VERBOSITY'] = '0'
warnings.filterwarnings('ignore', category=UserWarning, module='pytorch_lightning')
warnings.filterwarnings('ignore', category=FutureWarning, module='pytorch_lightning')
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning.utilities.rank_zero").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning.accelerators.cuda").setLevel(logging.ERROR)

print("="*90)
print("HITTER ROS FEATURE BUILDING & PREDICTION")
print("="*90)
print()

# Reload modules to ensure we have the latest version
import importlib
import new_pipeline.common.projections.usage_projections
import new_pipeline.common.projections.ros_projections
importlib.reload(new_pipeline.common.projections.usage_projections)
importlib.reload(new_pipeline.common.projections.ros_projections)

from new_pipeline.common.projections.usage_projections import (
    get_team_games_from_data,
    calculate_hitter_remaining_pa
)
from new_pipeline.common.projections.ros_projections import format_hitter_ros_display
from new_pipeline.common.data_preparation.clean_multi_team_data import clean_multi_team_players

hitter_2025_raw = load_current_season_data('hitter', year=2025)

# Clean multi-team players
hitter_2025_raw = clean_multi_team_players(hitter_2025_raw, year=2025, player_type="hitter")
print(f"Cleaned hitters - Multi-team players: {sum(hitter_2025_raw['Team'].str.contains(',', na=False))}")
hitter_2025_processed = run_data_pipeline(hitter_2025_raw, player_type='hitter')

# Get team-specific games played using utility function
team_games_dict, league_median_games = get_team_games_from_data(hitter_2025_processed)

# Calculate season progression
season_pct = league_median_games / 162
games_remaining = 162 - league_median_games

print(f"League median games: {league_median_games:.0f}, Remaining: {games_remaining:.0f}, Season: {season_pct:.1%}")
print(f"(Individual projections use team-specific games + multi-team handling)")
print()

# Get ALL qualified hitters (minimum PA threshold)
# Use 3.1 PA/G * 95 games = ~75 PA minimum
min_pa = 75
all_qualified_hitters = hitter_2025_processed[hitter_2025_processed['PA'] >= min_pa].copy()

print(f"Building ROS features for ALL qualified hitters (>={min_pa} PA)...")
print(f"  Found {len(all_qualified_hitters)} qualified hitters")

# Build ROS features for ALL qualified hitters
hitter_ros_features_all = hitter_builder.build_features_batch(
    current_season_df=all_qualified_hitters,
    historical_df=hitter_processed,
    injury_df=injury_data_historical
)

# Run complete ROS prediction workflow
from new_pipeline.common.projections.ros_projections import run_ros_predictions

results_hitters = run_ros_predictions(
    ensemble=hitter_ros,
    current_df=hitter_ros_features_all,
    historical_df=hitter_with_features,
    player_type='hitter',
    role='hitter',
    season_pct=season_pct,
    blend_ratio=0.70,
    team_games_dict=team_games_dict,
    league_median_games=league_median_games,
    use_percentile_tiers=True
)

# Unpack results
ros_predictions_all = results_hitters['predictions']
tier_labels_hitters_all = results_hitters['tiers']
projected_remaining_pa_all = results_hitters['usage_projected']
ros_display_all = results_hitters['display']

# Get thresholds for display
# Get percentile targets and actual cutoffs
from new_pipeline.models.ros.tier_thresholds import get_tier_percentiles
elite_pct, good_pct = get_tier_percentiles('hitter')
elite_cutoff = np.percentile(ros_predictions_all['mean'], 100 * (1 - elite_pct))
good_cutoff = np.percentile(ros_predictions_all['mean'], 100 * (1 - good_pct))

print(f"Tier Classification (Percentile-Based):")
print(f"  Elite: Top {elite_pct*100:.1f}% (~{int(len(all_qualified_hitters) * elite_pct)} of {len(all_qualified_hitters)} players)")
print(f"    -> Cutoff this run: {elite_cutoff:.2f} ROS WAR")
print(f"  Good: Top {good_pct*100:.1f}% (~{int(len(all_qualified_hitters) * good_pct)} of {len(all_qualified_hitters)} players)")
print(f"    -> Cutoff this run: {good_cutoff:.2f} ROS WAR")
print()


# Extract additional usage stats for display
hitter_team_games_all = all_qualified_hitters['Team'].map(team_games_dict).fillna(league_median_games).values
pa_per_game_all = all_qualified_hitters['PA'].values / np.maximum(all_qualified_hitters['G'].values, 1)
participation_rate_all = all_qualified_hitters['PA'].values / (all_qualified_hitters['G'].values * 4.5)  # Assume ~4.5 PA per game
current_pa_all = all_qualified_hitters['PA'].values


# Sort by ROS_WAR (descending) and take top 20
hitter_ros_ranking = np.argsort(ros_display_all['ros_war'])[::-1]
top_20_hitter_idx = hitter_ros_ranking[:20]

# Apply sorting to create top 20 arrays
top_20_hitters = all_qualified_hitters.iloc[top_20_hitter_idx].reset_index(drop=True)
tier_labels_hitters = tier_labels_hitters_all[top_20_hitter_idx]
projected_remaining_pa = projected_remaining_pa_all[top_20_hitter_idx]
hitter_team_games = hitter_team_games_all[top_20_hitter_idx]
pa_per_game = pa_per_game_all[top_20_hitter_idx]
participation_rate = participation_rate_all[top_20_hitter_idx]
current_pa = current_pa_all[top_20_hitter_idx]
ros_display = {
    'ros_war': ros_display_all['ros_war'][top_20_hitter_idx],
    'ros_rate': ros_display_all['ros_rate'][top_20_hitter_idx],
    'ros_q50': ros_display_all['ros_q50'][top_20_hitter_idx],
    'ros_q90': ros_display_all['ros_q90'][top_20_hitter_idx]
}

print(f"  Predicted all {len(all_qualified_hitters)} hitters, selected top 20 by ROS_WAR")
print()
print("Feature building & prediction complete!")
print("="*90)

HITTER ROS FEATURE BUILDING & PREDICTION

Loading partial season data: fangraphs_hitters_2025_firsthalf.csv
Found 25 multi-team hitter(s) to clean
Using cached stint data for player 17350 (2025)
  ✓ Rafael Devers: SFG, BOS (current: BOS)
Using cached stint data for player 20572 (2025)
  ✓ Kody Clemens: PHI, MIN (current: MIN)
Using cached stint data for player 15271 (2025)
  ✓ Austin Wynns: ATH, CIN (current: CIN)
Using cached stint data for player 19318 (2025)
  ✓ Matt Thaiss: CHW, TBR (current: TBR)
Using cached stint data for player 19458 (2025)
  ✓ Cooper Hummel: HOU, BAL (current: BAL)
Using cached stint data for player 17735 (2025)
  ✓ Tyler Alexander: CHW, MIL (current: MIL)
Using cached stint data for player 19262 (2025)
  ✓ Garrett Hampson: STL, ARI, CIN (current: CIN)
Using cached stint data for player 13768 (2025)
  ✓ Travis Jankowski: TBR, CHW, NYM (current: NYM)
Using cached stint data for player 16953 (2025)
  ✓ Chadwick Tromp: BAL, ATL (current: ATL)
Using cached stint d

17:45:34 - new_pipeline.common.transformers.filters - INFO - PAFilter: Removed 118 hitters with < 37 PA (partial season)
17:45:34 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Loaded Age for 673 hitters
17:45:34 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Added Age column (range: 21-41)
17:45:34 - new_pipeline.common.transformers.hitter_features - INFO - Loading hitter features...


Cleaned hitters - Multi-team players: 22


17:45:35 - new_pipeline.common.transformers.hitter_features - INFO - Loaded 11 hitter feature sets (35 total columns)
17:45:35 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Learned replacement values for 28 features
17:45:35 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Imputed 99 missing values
17:45:35 - new_pipeline.common.transformers.validators - WARNING - FeatureValidator found issues:
  - Feature 'AVG' range [0.07, 0.36] outside expected [0.1, 0.4]
  - Feature 'OBP' range [0.12, 0.47] outside expected [0.2, 0.5]
  - Feature 'SLG' range [0.07, 0.74] outside expected [0.2, 0.8]
17:45:35 - new_pipeline.common.transformers.feature_selector - INFO - FeatureSelector: Selected 9 features + 12 metadata columns
17:45:35 - new_pipeline.common.transformers.normalizers - INFO - WARNormalizer: Added 'WAR_per_600' column


League median games: 93, Remaining: 69, Season: 57.4%
(Individual projections use team-specific games + multi-team handling)

Building ROS features for ALL qualified hitters (>=75 PA)...
  Found 418 qualified hitters
  Player tiers: Tier1=203, Tier2=49, Tier3=166
Tier Classification (Percentile-Based):
  Elite: Top 3.0% (~12 of 418 players)
    -> Cutoff this run: 2.59 ROS WAR
  Good: Top 6.5% (~27 of 418 players)
    -> Cutoff this run: 2.23 ROS WAR

  Predicted all 418 hitters, selected top 20 by ROS_WAR

Feature building & prediction complete!


In [13]:
# Cell 9.6.1: Hitter ROS Diagnostic Table

# Reload table_utils to get latest version
import importlib
import new_pipeline.notebooks.shared.table_utils
importlib.reload(new_pipeline.notebooks.shared.table_utils)

from new_pipeline.notebooks.shared.table_utils import create_ros_diagnostic_table

print("="*90)
print("HITTERS (Top 20 by Projected ROS WAR)")
print("="*90)
print()

# Create diagnostic table
hitter_table = create_ros_diagnostic_table(
    names=top_20_hitters['Name'].values.tolist(),
    teams=top_20_hitters['Team'].values.tolist(),
    tiers=tier_labels_hitters.tolist(),
    usage_current=current_pa.tolist(),
    usage_projected=projected_remaining_pa.tolist(),
    ros_display={
        'ros_war': ros_display['ros_war'].tolist(),
        'ros_rate': ros_display['ros_rate'].tolist(),
        'ros_q50': ros_display['ros_q50'].tolist(),
        'ros_q90': ros_display['ros_q90'].tolist()
    },
    player_type='hitter',
    additional_cols={
        'G': top_20_hitters['G'].values.tolist(),
        'TeamG': hitter_team_games.tolist()
    }
)

print(hitter_table)
print()

# Tier distribution summary
elite_count = (tier_labels_hitters == 'elite').sum()
good_count = (tier_labels_hitters == 'good').sum()
avg_count = (tier_labels_hitters == 'average').sum()

print(f"Tier Distribution: Elite={elite_count}, Good={good_count}, Average={avg_count}")
print()
print("="*90)

HITTERS (Top 20 by Projected ROS WAR)

| Name                | Team | Tier  |  PA |  G | TeamG | Remaining_PA | Total_PA_Proj | ROS_Rate | ROS_WAR | ROS_q50 | ROS_q90 |
| :-------------------| :----| :-----|---: |--: |-----: |------------: |-------------: |--------: |-------: |-------: |-------: |
| Aaron Judge         | NYY  | elite | 429 | 96 |    96 |          295 |           724 |     9.12 |     4.5 |     4.5 |     6.1 |
| Cal Raleigh         | SEA  | elite | 417 | 94 |    95 |          294 |           711 |     8.11 |     4.0 |     4.0 |     5.7 |
| Bobby Witt Jr.      | KCR  | elite | 420 | 97 |    97 |          281 |           701 |     6.61 |     3.1 |     3.1 |     4.0 |
| Pete Crow-Armstrong | CHC  | elite | 401 | 95 |    95 |          283 |           684 |     6.52 |     3.1 |     3.1 |     4.4 |
| Jeremy Peña         | HOU  | elite | 350 | 82 |    93 |          260 |           610 |     6.87 |     3.0 |     3.0 |     3.9 |
| Shohei Ohtani       | LAD  | elite | 438 | 95 |  

In [14]:
# Cell 9.5.1: Starter ROS Diagnostic Table

# Reload table_utils to get latest version
import importlib
import new_pipeline.notebooks.shared.table_utils
importlib.reload(new_pipeline.notebooks.shared.table_utils)

from new_pipeline.notebooks.shared.table_utils import create_ros_diagnostic_table

print("="*90)
print("STARTERS (Top 20 by Projected ROS WAR)")
print("="*90)
print()

# Create diagnostic table
starter_table = create_ros_diagnostic_table(
    names=starters['Name'].values.tolist(),
    teams=starters['Team'].values.tolist(),
    tiers=tier_labels_starters.tolist(),
    usage_current=starters['IP'].values.tolist(),
    usage_projected=projected_remaining_ip.tolist(),
    ros_display={
        'ros_war': ros_display_starters['ros_war'].tolist(),
        'ros_rate': ros_display_starters['ros_rate'].tolist(),
        'ros_q50': ros_display_starters['ros_q50'].tolist(),
        'ros_q90': ros_display_starters['ros_q90'].tolist()
    },
    player_type='pitcher',
    role='starter',
    additional_cols={
        'Starts': starters['GS'].values.tolist(),
        'IP/Start': ip_per_start.tolist(),
        'TeamG': starter_team_games.tolist()
    }
)

print(starter_table)
print()

# Tier distribution summary
elite_count = (tier_labels_starters == 'elite').sum()
good_count = (tier_labels_starters == 'good').sum()
avg_count = (tier_labels_starters == 'average').sum()

print(f"Tier Distribution: Elite={elite_count}, Good={good_count}, Average={avg_count}")
print()
print("="*90)

STARTERS (Top 20 by Projected ROS WAR)

| Name               | Team | Tier  |  IP | Starts | IP/Start | TeamG | Proj_IP | ROS_Rate | ROS_WAR | ROS_q50 | ROS_q90 |
| :------------------| :----| :-----|---: |------: |--------: |-----: |-------: |--------: |-------: |-------: |-------: |
| Tarik Skubal       | DET  | elite | 121 |     19 |      6.4 |    95 |      85 |     5.88 |     3.1 |     3.1 |     3.4 |
| Paul Skenes        | PIT  | elite | 121 |     20 |      6.0 |    91 |      86 |     5.05 |     2.7 |     2.4 |     2.8 |
| Garrett Crochet    | BOS  | elite | 129 |     20 |      6.5 |    97 |      84 |     5.15 |     2.7 |     2.3 |     2.8 |
| Trevor Rogers      | BAL  | elite |  35 |      6 |      5.9 |    88 |      86 |     4.90 |     2.6 |     2.0 |     3.7 |
| Zack Wheeler       | PHI  | elite | 122 |     19 |      6.4 |    96 |      85 |     4.57 |     2.4 |     2.2 |     2.4 |
| Kris Bubic         | KCR  | elite | 108 |     18 |      6.0 |    97 |      78 |     4.43 |     2.

In [15]:
# Cell 9.5.2: Swing Pitcher ROS Diagnostic Table

# Reload table_utils to get latest version
import importlib
import new_pipeline.notebooks.shared.table_utils
importlib.reload(new_pipeline.notebooks.shared.table_utils)

from new_pipeline.notebooks.shared.table_utils import create_ros_diagnostic_table

print("="*90)
print("SWING PITCHERS (Top 20 by Projected ROS WAR)")
print("="*90)
print()

# Create diagnostic table
swing_table = create_ros_diagnostic_table(
    names=swing_pitchers['Name'].values.tolist(),
    teams=swing_pitchers['Team'].values.tolist(),
    tiers=tier_labels_swing.tolist(),
    usage_current=swing_pitchers['IP'].values.tolist(),
    usage_projected=projected_remaining_ip_swing.tolist(),
    ros_display={
        'ros_war': ros_display_swing['ros_war'].tolist(),
        'ros_rate': ros_display_swing['ros_rate'].tolist(),
        'ros_q50': ros_display_swing['ros_q50'].tolist(),
        'ros_q90': ros_display_swing['ros_q90'].tolist()
    },
    player_type='pitcher',
    role='swing',
    additional_cols={
        'G': swing_pitchers['G'].values.tolist(),
        'GS': swing_pitchers['GS'].values.tolist(),
        'IP/G': ip_per_appearance_swing.tolist(),
        'TeamG': swing_team_games.tolist()
    }
)

print(swing_table)
print()

# Tier distribution summary
elite_count = (tier_labels_swing == 'elite').sum()
good_count = (tier_labels_swing == 'good').sum()
avg_count = (tier_labels_swing == 'average').sum()

print(f"Tier Distribution: Elite={elite_count}, Good={good_count}, Average={avg_count}")
print()
print("="*90)

SWING PITCHERS (Top 20 by Projected ROS WAR)

| Name              | Team     | Tier    | IP |  G | IP/G | TeamG | Proj_IP | ROS_Rate | ROS_WAR | ROS_q50 | ROS_q90 |
| :-----------------| :--------| :-------|--: |--: |----: |-----: |-------: |--------: |-------: |-------: |-------: |
| Janson Junk       | MIA      | elite   | 50 | 10 | 5.01 |    91 |      39 |     3.08 |     0.7 |     0.7 |     0.9 |
| Jack Dreyer       | LAD      | elite   | 48 | 37 | 1.30 |    95 |      34 |     3.30 |     0.7 |     0.6 |     0.8 |
| Joe Boyle         | TBR      | elite   | 14 |  3 | 4.67 |    91 |      11 |     8.06 |     0.5 |     0.5 |     0.7 |
| Eric Lauer        | TOR      | elite   | 55 | 14 | 3.93 |    94 |      40 |     2.20 |     0.5 |     0.5 |     0.8 |
| Ben Casparius     | LAD      | good    | 62 | 28 | 2.22 |    95 |      44 |     1.87 |     0.5 |     0.5 |     0.7 |
| Ryne Nelson       | ARI      | good    | 78 | 20 | 3.90 |    96 |      54 |     1.53 |     0.5 |     0.6 |     1.0 |
| 

In [16]:
# Cell 9.5.3: Reliever ROS Diagnostic Table

print("="*90)
print("RELIEVERS (Top 20 by Projected ROS WAR)")
print("="*90)
print()

# Create diagnostic table
reliever_table = create_ros_diagnostic_table(
    names=relievers['Name'].values.tolist(),
    teams=relievers['Team'].values.tolist(),
    tiers=tier_labels_relievers.tolist(),
    usage_current=relievers['IP'].values.tolist(),
    usage_projected=projected_remaining_ip_rel.tolist(),
    ros_display={
        'ros_war': ros_display_relievers['ros_war'].tolist(),
        'ros_rate': ros_display_relievers['ros_rate'].tolist(),
        'ros_q50': ros_display_relievers['ros_q50'].tolist(),
        'ros_q90': ros_display_relievers['ros_q90'].tolist()
    },
    player_type='pitcher',
    role='reliever',
    additional_cols={
        'G': relievers['G'].values.tolist(),
        'IP/G': ip_per_appearance.tolist(),
        'App_Rate': appearance_rate.tolist(),
        'TeamG': reliever_team_games.tolist()
    }
)

print(reliever_table)
print()

# Tier distribution summary
elite_count = (tier_labels_relievers == 'elite').sum()
good_count = (tier_labels_relievers == 'good').sum()
avg_count = (tier_labels_relievers == 'average').sum()

print(f"Tier Distribution: Elite={elite_count}, Good={good_count}, Average={avg_count}")
print()
print("="*90)

RELIEVERS (Top 20 by Projected ROS WAR)

| Name             | Team | Tier    | IP |  G | IP/G | App_Rate | TeamG | Proj_IP | ROS_Rate | ROS_WAR | ROS_q50 | ROS_q90 |
| :----------------| :----| :-------|--: |--: |----: |--------: |-----: |-------: |--------: |-------: |-------: |-------: |
| Aroldis Chapman  | BOS  | elite   | 38 | 41 | 0.93 |     0.42 |    97 |      25 |     2.00 |     1.1 |     1.1 |     1.5 |
| Griffin Jax      | MIN  | elite   | 41 | 44 | 0.93 |     0.48 |    92 |      31 |     1.34 |     0.9 |     0.9 |     1.0 |
| Cade Smith       | CLE  | elite   | 41 | 43 | 0.95 |     0.47 |    92 |      31 |     1.26 |     0.8 |     0.8 |     0.9 |
| Jhoan Duran      | MIN  | elite   | 43 | 44 | 0.98 |     0.48 |    92 |      33 |     1.17 |     0.8 |     0.8 |     1.1 |
| Edwin Díaz       | NYM  | elite   | 38 | 37 | 1.03 |     0.38 |    97 |      25 |     1.34 |     0.7 |     0.7 |     1.2 |
| Emmanuel Clase   | CLE  | elite   | 43 | 43 | 1.00 |     0.47 |    92 |      33 | 